# TBX11K — Object Detection Pipeline (Colab)
## 4 Architectures: FCOS, EfficientDet-D2, RetinaNet, DETR

**Prerequisite**: Upload TBX11K dataset as `images.zip` to `MyDrive/OBJECT_DETECTION/images.zip` in Google Drive.

**Run order**:
1. Cell 1 (setup + data)
2. Cells 2-14 (write all Python files)
3. Cell 15 (convert + EDA)
4. Cells 16-19 (train each model)
5. Cell 20 (comparison)
6. Cell 21 (confusion matrices + XAI)


In [ ]:
# Cell 1: Setup
import os
from google.colab import drive
drive.mount('/content/drive')

!cd /content && rm -rf /content/OBJECT_DETECTION 2>/dev/null
!mkdir -p /content/OBJECT_DETECTION
!cp /content/drive/MyDrive/OBJECT_DETECTION/images.zip /content/images.zip
!unzip -qo /content/images.zip -d /content/OBJECT_DETECTION/

# Wrap train/val/test into Images/ (zip has no Images wrapper)
if os.path.isdir('/content/OBJECT_DETECTION/train'):
    !mkdir -p /content/OBJECT_DETECTION/Images
    !mv /content/OBJECT_DETECTION/train /content/OBJECT_DETECTION/Images/
    !mv /content/OBJECT_DETECTION/val /content/OBJECT_DETECTION/Images/
    !mv /content/OBJECT_DETECTION/test /content/OBJECT_DETECTION/Images/

# If zip had Images/ wrapper instead, move contents up
if os.path.isdir('/content/OBJECT_DETECTION/Images/Images'):
    !mv /content/OBJECT_DETECTION/Images/Images/* /content/OBJECT_DETECTION/Images/
    !rmdir /content/OBJECT_DETECTION/Images/Images

# Handle lowercase 'images/' (case-sensitive filesystem)
if os.path.isdir('/content/OBJECT_DETECTION/images') and not os.path.isdir('/content/OBJECT_DETECTION/Images'):
    !mv /content/OBJECT_DETECTION/images /content/OBJECT_DETECTION/Images

# Change to project directory (build-mode safe)
%cd /content/OBJECT_DETECTION
os.makedirs('results/eda', exist_ok=True)
os.makedirs('utils', exist_ok=True)
os.makedirs('explain', exist_ok=True)

!pip install -q torch>=2.0.0 torchvision>=0.15.0 effdet timm pycocotools pillow tqdm matplotlib seaborn
import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Debug: show directory structure to verify data
import subprocess
result = subprocess.run(['find', '.', '-maxdepth', '3', '-type', 'd'], capture_output=True, text=True, timeout=10)
print('\nDirectory structure (first 3 levels):')
print(result.stdout[:1500])

### Cell 2: Write `utils/__init__.py`


In [ ]:
%%writefile utils/__init__.py


### Cell 3: Write `utils/coco_dataset.py`


In [ ]:
%%writefile utils/coco_dataset.py
import os
from PIL import Image
import torch
from torch.utils.data import Dataset
from pycocotools.coco import COCO
import torchvision.transforms as T

class CocoDetection(Dataset):
    def __init__(self, root, ann_file, transforms=None):
        self.root = root
        self.transforms = transforms
        self.coco = COCO(ann_file)
        self.ids = sorted(self.coco.imgs.keys())

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        img_info = self.coco.loadImgs(img_id)[0]
        path = os.path.join(self.root, img_info['file_name'])
        image = Image.open(path).convert('RGB')
        boxes, labels, area, iscrowd = [], [], [], []
        for ann in anns:
            bbox = ann['bbox']
            boxes.append([bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]])
            labels.append(ann['category_id'])
            area.append(ann['area'])
            iscrowd.append(ann.get('iscrowd', 0))
        if boxes:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            area = torch.as_tensor(area, dtype=torch.float32)
            iscrowd = torch.as_tensor(iscrowd, dtype=torch.uint8)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.uint8)
        target = {
            'boxes': boxes, 'labels': labels, 'image_id': torch.tensor([img_id]),
            'area': area, 'iscrowd': iscrowd
        }
        if self.transforms is not None:
            image, target = self.transforms(image, target)
        return image, target

    def __len__(self):
        return len(self.ids)


def get_transform(train):
    transforms_list = [T.ToTensor()]
    if train:
        transforms_list.append(T.RandomHorizontalFlip(0.5))
    transform = T.Compose(transforms_list)
    def _apply(image, target):
        return transform(image), target
    return _apply


def collate_fn(batch):
    return tuple(zip(*batch))

### Cell 4: Write `utils/engine.py`


In [ ]:
%%writefile utils/engine.py
import os, json, time
import torch
import numpy as np
from torchvision.ops import box_iou
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

CLASS_NAMES = ['Background', 'ActiveTuberculosis', 'ObsoletePulmonaryTuberculosis']


def train_one_epoch(model, optimizer, data_loader, device, epoch, scaler=None, print_freq=50, clip_norm=None):
    model.train()
    total_loss = 0
    start_time = time.time()
    for i, (images, targets) in enumerate(data_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        if scaler is not None:
            with torch.amp.autocast(device_type='cuda'):
                loss_dict = model(images, targets)
                losses = sum(loss_dict.values())
            scaler.scale(losses).backward()
            if clip_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss_dict = model(images, targets)
            losses = sum(loss_dict.values())
            optimizer.zero_grad()
            losses.backward()
            if clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            optimizer.step()
        optimizer.zero_grad()
        total_loss += losses.item()
        if i % print_freq == 0:
            elapsed = time.time() - start_time
            print(f'  Epoch [{epoch}] Step [{i}/{len(data_loader)}] Loss: {losses.item():.4f} | Time: {elapsed:.1f}s')
    avg_loss = total_loss / len(data_loader)
    print(f'  Epoch [{epoch}] Avg Loss: {avg_loss:.4f}')
    return avg_loss


@torch.no_grad()
def evaluate(model, data_loader, device, output_file=None):
    model.eval()
    coco_gt = data_loader.dataset.coco
    results = []
    img_ids = []
    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)
        for target, output in zip(targets, outputs):
            img_id = target['image_id'].item()
            img_ids.append(img_id)
            boxes = output['boxes'].cpu().numpy()
            scores = output['scores'].cpu().numpy()
            labels = output['labels'].cpu().numpy()
            for box, score, label in zip(boxes, scores, labels):
                x1, y1, x2, y2 = box
                w, h = x2 - x1, y2 - y1
                results.append({
                    'image_id': img_id, 'bbox': [float(x1), float(y1), float(w), float(h)],
                    'score': float(score), 'category_id': int(label)
                })
    if not results:
        print('[EVAL] No predictions.')
        return None
    coco_dt = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    coco_eval.params.imgIds = sorted(set(img_ids))
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    stats = coco_eval.stats.tolist()
    metrics = {
        'mAP@0.5:0.95': stats[0], 'mAP@0.5': stats[1], 'mAP@0.75': stats[2],
        'mAP_small': stats[3], 'mAP_medium': stats[4], 'mAP_large': stats[5],
        'AR@1': stats[6], 'AR@10': stats[7], 'AR@100': stats[8],
    }
    if output_file:
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        with open(output_file, 'w') as f:
            json.dump({'metrics': metrics, 'predictions': results}, f, indent=2)
    return coco_eval


@torch.no_grad()
def evaluate_test(model, data_loader, device, output_file):
    model.eval()
    results = []
    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)
        for target, output in zip(targets, outputs):
            img_id = target['image_id'].item()
            boxes = output['boxes'].cpu().numpy()
            scores = output['scores'].cpu().numpy()
            labels = output['labels'].cpu().numpy()
            for box, score, label in zip(boxes, scores, labels):
                x1, y1, x2, y2 = box
                w, h = x2 - x1, y2 - y1
                results.append({
                    'image_id': img_id, 'bbox': [float(x1), float(y1), float(w), float(h)],
                    'score': float(score), 'category_id': int(label)
                })
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'[TEST] {len(results)} predictions saved')


@torch.no_grad()
def compute_confusion_matrix(model, data_loader, device, iou_thresh=0.5, score_thresh=0.05):
    model.eval()
    confusion = np.zeros((3, 3), dtype=int)
    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        outputs = model(images)
        for pred, target in zip(outputs, targets):
            gt_boxes = target['boxes'].cpu()
            gt_labels = target['labels'].cpu()
            keep = pred['scores'] >= score_thresh
            pred_boxes = pred['boxes'][keep].cpu()
            pred_labels = pred['labels'][keep].cpu()
            matched_gt, matched_pred = set(), set()
            if len(gt_boxes) > 0 and len(pred_boxes) > 0:
                iou_matrix = box_iou(gt_boxes, pred_boxes)
                for gt_idx in range(len(gt_boxes)):
                    best_pred, best_iou = -1, iou_thresh
                    for pred_idx in range(len(pred_boxes)):
                        if pred_idx not in matched_pred:
                            iou = iou_matrix[gt_idx, pred_idx].item()
                            if iou > best_iou:
                                best_iou, best_pred = iou, pred_idx
                    if best_pred >= 0:
                        confusion[gt_labels[gt_idx].item(), pred_labels[best_pred].item()] += 1
                        matched_gt.add(gt_idx)
                        matched_pred.add(best_pred)
                    else:
                        confusion[gt_labels[gt_idx].item(), 0] += 1
                        matched_gt.add(gt_idx)
            for gt_idx in range(len(gt_boxes)):
                if gt_idx not in matched_gt:
                    confusion[gt_labels[gt_idx].item(), 0] += 1
            for pred_idx in range(len(pred_boxes)):
                if pred_idx not in matched_pred:
                    confusion[0, pred_labels[pred_idx].item()] += 1
    return confusion


def save_confusion_matrix_plot(confusion, output_path, class_names=None):
    if class_names is None:
        class_names = CLASS_NAMES
    fig, ax = plt.subplots(figsize=(7, 6))
    total_per_row = confusion.sum(axis=1, keepdims=True)
    total_per_row = np.where(total_per_row == 0, 1, total_per_row)
    confusion_norm = confusion / total_per_row
    sns.heatmap(confusion_norm, annot=confusion, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax,
                cbar_kws={'label': 'Fraction'})
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Ground Truth')
    ax.set_title('Confusion Matrix')
    plt.tight_layout()
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    plt.savefig(output_path, dpi=150)
    plt.close()
    print(f'[CONFUSION] Saved to {output_path}')

### Cell 5: Write `explain/__init__.py`


In [ ]:
%%writefile explain/__init__.py


### Cell 6: Write `explain/gradcam.py`


In [ ]:
%%writefile explain/gradcam.py
import torch
import torch.nn.functional as F
import numpy as np

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self._activations = None
        self._gradients = None
        self._forward_handle = target_layer.register_forward_hook(self._hook_forward)
        self._backward_handle = target_layer.register_full_backward_hook(self._hook_backward)
    def _hook_forward(self, module, input, output):
        self._activations = output.detach()
    def _hook_backward(self, module, grad_input, grad_output):
        self._gradients = grad_output[0].detach()
    def remove_hooks(self):
        self._forward_handle.remove()
        self._backward_handle.remove()
    def generate(self, image, target_class=None):
        self.model.zero_grad()
        if image.grad is not None:
            image.grad.zero_()
        predictions = self.model(image)
        if target_class is None:
            if len(predictions[0]['scores']) == 0:
                return None
            best_idx = predictions[0]['scores'].argmax()
            target_class = predictions[0]['labels'][best_idx].item()
        loss = torch.tensor(0.0, device=image.device)
        for pred in predictions:
            for label, score in zip(pred['labels'], pred['scores']):
                if label.item() == target_class:
                    loss = loss + score
        if loss == 0:
            return None
        loss.backward()
        if self._gradients is None or self._activations is None:
            return None
        weights = self._gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self._activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=image.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


def get_target_layer(model, architecture):
    if architecture in ('fcos', 'retinanet'):
        return model.backbone.body.layer4
    elif architecture == 'efficientdet':
        return model.backbone.conv_head
    raise ValueError(f'Unsupported: {architecture}')

### Cell 7: Write `explain/detr_attention.py`


In [ ]:
%%writefile explain/detr_attention.py
import torch
import torch.nn.functional as F
import numpy as np

class DETRAttentionExtractor:
    def __init__(self, model):
        self.model = model
        self._attention_weights = None
        decoder_layer = model.transformer.decoder.layers[-1]
        self._handle = decoder_layer.multihead_attn.register_forward_hook(self._hook_attention)
    def _hook_attention(self, module, input, output):
        q, k = input[0], input[1]
        attn_weights = torch.bmm(q.transpose(0, 1), k.transpose(0, 1).transpose(1, 2))
        attn_weights = F.softmax(attn_weights / (q.size(-1) ** 0.5), dim=-1)
        self._attention_weights = attn_weights.detach()
    def remove_hook(self):
        self._handle.remove()
    def extract(self, image):
        self.model.zero_grad()
        with torch.no_grad():
            output = self.model(image)
        if self._attention_weights is None:
            return None, output
        attn = self._attention_weights.mean(dim=0)
        pred_boxes = output[0]['boxes'].cpu().numpy()
        pred_scores = output[0]['scores'].cpu().numpy()
        pred_labels = output[0]['labels'].cpu().numpy()
        heatmaps, meta = [], []
        for q_idx in range(min(attn.shape[0], 100)):
            if q_idx >= len(pred_scores) or pred_scores[q_idx] < 0.1:
                continue
            attn_map = attn[q_idx].reshape(-1).cpu().numpy()
            h = w = int(np.sqrt(len(attn_map)))
            if h * w != len(attn_map):
                h = w = 32
            attn_map = attn_map[:h*w].reshape(h, w)
            attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)
            heatmaps.append(attn_map)
            meta.append({'query_idx': q_idx, 'score': float(pred_scores[q_idx]),
                        'label': int(pred_labels[q_idx]), 'box': pred_boxes[q_idx].tolist()})
        return heatmaps, meta

### Cell 8: Write `explain/visualize.py`


In [ ]:
%%writefile explain/visualize.py
import os
import numpy as np
from PIL import Image, ImageDraw
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

CLASS_NAMES = {0: 'Background', 1: 'ActiveTuberculosis', 2: 'ObsoletePulmonaryTuberculosis'}
CLASS_COLORS = {1: (255, 0, 0), 2: (0, 200, 255)}


def overlay_heatmap(image_tensor, heatmap, alpha=0.6):
    img_np = image_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
    img_np = (img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
    img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)
    if heatmap is None:
        return Image.fromarray(img_np)
    cmap = plt.get_cmap('jet')
    heatmap_colored = (cmap(heatmap)[:, :, :3] * 255).astype(np.uint8)
    blended = np.clip((1 - alpha) * img_np + alpha * heatmap_colored, 0, 255).astype(np.uint8)
    return Image.fromarray(blended)


def draw_detections(pil_image, predictions, score_thresh=0.3):
    draw = ImageDraw.Draw(pil_image)
    for box, score, label in zip(predictions.get('boxes', []),
                                 predictions.get('scores', []),
                                 predictions.get('labels', [])):
        if score < score_thresh:
            continue
        x1, y1, x2, y2 = box[:4]
        color = CLASS_COLORS.get(int(label), (255, 255, 0))
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        draw.text((x1 + 2, y1 - 16), f'{CLASS_NAMES.get(int(label), str(label))}: {score:.2f}', fill=color)
    return pil_image

### Cell 9: Write `convert.py`


In [ ]:
%%writefile convert.py
import os, json, glob, shutil, argparse

DATA_ROOT = 'Images'
OUTPUT_DIR = 'dataset'
CLASS_MAP = {'ActiveTuberculosis': 1, 'ObsoletePulmonaryTuberculosis': 2}

def convert_to_coco(splits):
    for split in splits:
        images, annotations = [], []
        ann_id = 0
        ann_dir = os.path.join(DATA_ROOT, split, 'ann')
        img_dir = os.path.join(DATA_ROOT, split, 'img')
        out_img_dir = os.path.join(OUTPUT_DIR, 'coco', split)
        os.makedirs(out_img_dir, exist_ok=True)
        for f in sorted(glob.glob(os.path.join(ann_dir, '*.json'))):
            with open(f) as fh:
                d = json.load(fh)
            basename = os.path.basename(f).replace('.png.json', '.png')
            img_id = len(images)
            images.append({
                'id': img_id, 'file_name': basename,
                'width': d['size']['width'], 'height': d['size']['height']
            })
            src_path = os.path.join(img_dir, basename)
            dst_path = os.path.join(out_img_dir, basename)
            if not os.path.exists(dst_path):
                shutil.copy2(src_path, dst_path)
            for obj in d.get('objects', []):
                ext = obj['points']['exterior']
                x1, y1 = ext[0]
                x2, y2 = ext[1]
                w, h = x2 - x1, y2 - y1
                if w <= 0 or h <= 0:
                    continue
                annotations.append({
                    'id': ann_id, 'image_id': img_id,
                    'bbox': [x1, y1, w, h], 'area': w * h,
                    'category_id': CLASS_MAP[obj['classTitle']], 'iscrowd': 0
                })
                ann_id += 1
        out = {
            'images': images,
            'annotations': annotations,
            'categories': [
                {'id': 1, 'name': 'ActiveTuberculosis'},
                {'id': 2, 'name': 'ObsoletePulmonaryTuberculosis'},
            ],
        }
        with open(os.path.join(OUTPUT_DIR, 'coco', f'{split}.json'), 'w') as f:
            json.dump(out, f)
        print(f'[COCO] {split}: {len(images)} images, {len(annotations)} annotations')

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--data-root', type=str, default='Images')
    parser.add_argument('--output-dir', type=str, default='dataset')
    args = parser.parse_args()
    global DATA_ROOT, OUTPUT_DIR
    DATA_ROOT, OUTPUT_DIR = args.data_root, args.output_dir
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    convert_to_coco(['train', 'val', 'test'])
    print('Conversion complete.')

if __name__ == '__main__':
    main()

### Cell 10: Write `eda.py`


In [ ]:
%%writefile eda.py
import os, json, glob, random
from collections import Counter
from statistics import mean, median
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 150})
DATA_ROOT = 'Images'
OUTPUT_DIR = 'results/eda'
os.makedirs(OUTPUT_DIR, exist_ok=True)
CLASS_MAP = {'ActiveTuberculosis': 1, 'ObsoletePulmonaryTuberculosis': 2}

def load_annotations(splits=None):
    if splits is None:
        splits = ['train', 'val', 'test']
    records = []
    for split in splits:
        ann_dir = os.path.join(DATA_ROOT, split, 'ann')
        img_dir = os.path.join(DATA_ROOT, split, 'img')
        for f in sorted(glob.glob(os.path.join(ann_dir, '*.json'))):
            with open(f) as fh:
                d = json.load(fh)
            basename = os.path.basename(f).replace('.png.json', '')
            tags = [t['name'] for t in d.get('tags', [])]
            objs = []
            for o in d.get('objects', []):
                ext = o['points']['exterior']
                x1, y1 = ext[0]
                x2, y2 = ext[1]
                objs.append({
                    'class': o['classTitle'], 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                    'w': x2-x1, 'h': y2-y1, 'cx': (x1+x2)/2, 'cy': (y1+y2)/2,
                    'area': (x2-x1)*(y2-y1)
                })
            records.append({
                'split': split, 'img_id': basename,
                'img_path': os.path.join(img_dir, f'{basename}.png'),
                'tags': tags, 'objs': objs,
                'width': d['size']['width'], 'height': d['size']['height']
            })
    return records

def plot_tag_distribution(records):
    splits, tag_order = ['train', 'val', 'test'], ['healthy', 'sick_but_non-tb', 'active_tb', 'latent_tb', 'active&latent_tb']
    data = {}
    for sp in splits:
        c = Counter()
        for r in records:
            if r['split'] == sp:
                for t in r['tags']:
                    c[t] += 1
        data[sp] = [c.get(t, 0) for t in tag_order]
    x, w = np.arange(len(tag_order)), 0.25
    fig, ax = plt.subplots(figsize=(12, 6))
    for i, sp in enumerate(splits):
        bars = ax.bar(x + i*w, data[sp], w, label=sp.upper(), color=['#4C72B0','#DD8452','#55A868'][i])
        for bar, val in zip(bars, data[sp]):
            if val > 0:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10, str(val), ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x + w)
    ax.set_xticklabels([t.replace('_','\n') for t in tag_order])
    ax.set_ylabel('Image count')
    ax.set_title('Tag Distribution per Split')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/tag_distribution.png')
    plt.close()

def plot_bbox_class_distribution(records):
    splits, classes = ['train', 'val'], ['ActiveTuberculosis', 'ObsoletePulmonaryTuberculosis']
    data = {}
    for sp in splits:
        c = Counter()
        for r in records:
            if r['split'] == sp:
                for o in r['objs']:
                    c[o['class']] += 1
        data[sp] = [c.get(cls, 0) for cls in classes]
    x, w = np.arange(len(classes)), 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, sp in enumerate(splits):
        bars = ax.bar(x + i*w, data[sp], w, label=sp.upper(), color=['#4C72B0','#DD8452'][i])
        for bar, val in zip(bars, data[sp]):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, str(val), ha='center', va='bottom', fontsize=10)
    ax.set_xticks(x + w/2)
    ax.set_xticklabels(classes, rotation=15, ha='right')
    ax.set_ylabel('Bounding box count')
    ax.set_title('Object Class Distribution')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/bbox_class_distribution.png')
    plt.close()

def plot_bbox_spatial_heatmap(records):
    all_cx, all_cy = [], []
    for r in records:
        if r['split'] == 'test':
            continue
        for o in r['objs']:
            all_cx.append(o['cx']/r['width'])
            all_cy.append(o['cy']/r['height'])
    if not all_cx:
        return
    fig, ax = plt.subplots(figsize=(6, 6))
    heatmap, _, _ = np.histogram2d(all_cx, all_cy, bins=32, range=[[0,1],[0,1]])
    ax.imshow(heatmap.T, origin='lower', cmap='hot', interpolation='bilinear', extent=[0,1,0,1])
    ax.set_xlabel('Normalized X')
    ax.set_ylabel('Normalized Y')
    ax.set_title('BBox Center Distribution')
    plt.colorbar(plt.cm.ScalarMappable(cmap='hot'), ax=ax, label='Count')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/bbox_spatial_heatmap.png')
    plt.close()

def plot_bbox_size_distribution(records):
    widths, heights = [], []
    for r in records:
        if r['split'] == 'test':
            continue
        for o in r['objs']:
            widths.append(o['w'])
            heights.append(o['h'])
    if not widths:
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(widths, heights, alpha=0.4, s=15, c='#4C72B0', edgecolors='none')
    axes[0].set_xlabel('Width (px)'), axes[0].set_ylabel('Height (px)')
    axes[0].set_title(f'Box Size (n={len(widths)})')
    axes[1].hist([(w*h)/(512*512)*100 for w,h in zip(widths,heights)], bins=40, color='#DD8452', edgecolor='white', alpha=0.8)
    axes[1].set_xlabel('Relative area (%)'), axes[1].set_ylabel('Count'), axes[1].set_title('Box Area Distribution')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/bbox_size_distribution.png')
    plt.close()

def plot_bbox_aspect_ratio(records):
    ratios = []
    for r in records:
        if r['split'] == 'test':
            continue
        for o in r['objs']:
            if o['h'] > 0:
                ratios.append(o['w']/o['h'])
    if not ratios:
        return
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(ratios, bins=30, color='#55A868', edgecolor='white', alpha=0.8)
    ax.axvline(mean(ratios), color='red', linestyle='--', alpha=0.7, label=f'Mean={mean(ratios):.2f}')
    ax.axvline(1.0, color='gray', linestyle='-', alpha=0.4, label='Square')
    ax.set_xlabel('Aspect Ratio (W/H)'), ax.set_ylabel('Count'), ax.set_title('BBox Aspect Ratio')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/bbox_aspect_ratio.png')
    plt.close()

def plot_boxes_per_image(records):
    splits = ['train', 'val']
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for idx, sp in enumerate(splits):
        c = Counter()
        for r in records:
            if r['split'] == sp:
                c[len(r['objs'])] += 1
        bins = list(range(max(c.keys())+2))
        counts = [c.get(i,0) for i in range(max(c.keys())+2)]
        axes[idx].bar(bins, counts, color='#4C72B0', edgecolor='white', width=0.8)
        axes[idx].set_xlabel('Number of boxes'), axes[idx].set_ylabel('Count'), axes[idx].set_title(f'{sp.upper()} — Boxes per Image')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/boxes_per_image.png')
    plt.close()

def plot_sample_grid(records):
    pos = [r for r in records if r['objs'] and r['split'] != 'test']
    if len(pos) > 16:
        random.seed(42)
        pos = random.sample(pos, 16)
    n, cols = len(pos), 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*3.5, rows*3.5))
    axes = axes.flatten() if n > 1 else [axes]
    for i, r in enumerate(pos):
        img = Image.open(r['img_path']).convert('RGB')
        draw = ImageDraw.Draw(img, 'RGBA')
        for o in r['objs']:
            color = (255,0,0,120) if o['class']=='ActiveTuberculosis' else (0,255,170,120)
            draw.rectangle([o['x1'],o['y1'],o['x2'],o['y2']], outline=color[:3], width=3)
            draw.text((o['x1']+2, o['y1']-14), 'ATB' if o['class']=='ActiveTuberculosis' else 'OTB', fill=color[:3])
        axes[i].imshow(np.array(img))
        axes[i].set_title(f'{r["split"]}: {", ".join(r["tags"])}', fontsize=9)
        axes[i].axis('off')
    for j in range(n, len(axes)):
        axes[j].axis('off')
    plt.suptitle('Sample Images (Red=ActiveTB, Cyan=ObsoleteTB)', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/sample_grid.png')
    plt.close()

def print_dataset_summary(records):
    total = len(records)
    splits = Counter(r['split'] for r in records)
    total_boxes = sum(len(r['objs']) for r in records)
    tag_counter, box_counter = Counter(), Counter()
    for r in records:
        for t in r['tags']:
            tag_counter[t] += 1
        for o in r['objs']:
            box_counter[o['class']] += 1
    lines = ['='*60, 'DATASET SUMMARY — TBX11K', '='*60,
             f'Total images: {total}',
             f'  Train: {splits.get("train",0)}',
             f'  Val: {splits.get("val",0)}',
             f'  Test: {splits.get("test",0)}',
             f'Image size: 512x512', '',
             f'Total boxes: {total_boxes}', '',
             'Tag distribution:']
    for tag, count in tag_counter.most_common():
        lines.append(f'  {tag}: {count} ({count/total*100:.1f}%)')
    lines.extend(['', 'Box class distribution:'])
    for cls_name, count in box_counter.most_common():
        lines.append(f'  {cls_name}: {count} ({count/total_boxes*100:.1f}%)')
    summary = '\n'.join(lines)
    print(summary)
    with open(f'{OUTPUT_DIR}/dataset_summary.txt', 'w') as f:
        f.write(summary)

def main():
    records = load_annotations()
    print(f'Loaded {len(records)} annotation files.')
    print_dataset_summary(records)
    plot_tag_distribution(records)
    plot_bbox_class_distribution(records)
    plot_bbox_spatial_heatmap(records)
    plot_bbox_size_distribution(records)
    plot_bbox_aspect_ratio(records)
    plot_boxes_per_image(records)
    plot_sample_grid(records)
    print(f'EDA outputs saved to {OUTPUT_DIR}/')

if __name__ == '__main__':
    main()

### Cell 11: Write `train_fcos.py`


In [ ]:
%%writefile train_fcos.py
import os, sys, json
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.models.detection import fcos_resnet50_fpn

sys.path.insert(0, os.path.dirname(__file__))
from utils.coco_dataset import CocoDetection, get_transform, collate_fn
from utils.engine import train_one_epoch, evaluate, evaluate_test, compute_confusion_matrix, save_confusion_matrix_plot
from explain.gradcam import GradCAM, get_target_layer
from explain.visualize import overlay_heatmap, draw_detections

RESULTS_DIR = 'results/fcos'
os.makedirs(f'{RESULTS_DIR}/explain', exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/weights', exist_ok=True)

def get_weighted_sampler(dataset, pos_weight=4.0):
    weights = []
    for i in range(len(dataset)):
        _, target = dataset[i]
        weights.append(pos_weight if len(target['boxes']) > 0 else 1.0)
    return WeightedRandomSampler(weights, len(weights), replacement=True)

def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'[FCOS] Device: {device}')
    train_dataset = CocoDetection('dataset/coco/train', 'dataset/coco/train.json', transforms=get_transform(train=True))
    val_dataset = CocoDetection('dataset/coco/val', 'dataset/coco/val.json', transforms=get_transform(train=False))
    train_loader = DataLoader(train_dataset, batch_size=8, sampler=get_weighted_sampler(train_dataset), collate_fn=collate_fn, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)
    model = fcos_resnet50_fpn(weights_backbone="DEFAULT", num_classes=3).to(device)

    optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=0.001, momentum=0.9, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
    best_map, best_epoch = 0.0, -1
    for epoch in range(1, 101):
        train_one_epoch(model, optimizer, train_loader, device, epoch, scaler=scaler, clip_norm=1.0)
        scheduler.step()
        coco_eval = evaluate(model, val_loader, device)
        if coco_eval is not None and coco_eval.stats[0] > best_map:
            best_map, best_epoch = coco_eval.stats[0], epoch
            torch.save(model.state_dict(), f'{RESULTS_DIR}/weights/best_model.pth')
            print(f'  New best mAP={best_map:.4f} (epoch {epoch})')
    print(f'[FCOS] Best mAP: {best_map:.4f} at epoch {best_epoch}')

def evaluate_model():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    val_loader = DataLoader(CocoDetection('dataset/coco/val', 'dataset/coco/val.json', transforms=get_transform(train=False)), batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)
    model = fcos_resnet50_fpn(weights_backbone="DEFAULT", num_classes=3)
    model.load_state_dict(torch.load(f'{RESULTS_DIR}/weights/best_model.pth', map_location=device, weights_only=True))
    model.to(device).eval()
    coco_eval = evaluate(model, val_loader, device, output_file=f'{RESULTS_DIR}/val_preds.json')
    if coco_eval is not None:
        metrics = {
            'mAP@0.5:0.95': coco_eval.stats[0],
            'mAP@0.5': coco_eval.stats[1],
            'AP_ActiveTB': coco_eval.eval['precision'][0,:,1,0,2].mean(),
            'AP_ObsoleteTB': coco_eval.eval['precision'][0,:,2,0,2].mean(),
        }
        with open(f'{RESULTS_DIR}/metrics.json', 'w') as f:
            json.dump(metrics, f, indent=2)
    confusion = compute_confusion_matrix(model, val_loader, device)
    save_confusion_matrix_plot(confusion, f'{RESULTS_DIR}/confusion_matrix.png')
    if os.path.exists('dataset/coco/test.json'):
        test_loader = DataLoader(CocoDetection('dataset/coco/test', 'dataset/coco/test.json', transforms=get_transform(train=False)), batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)
        evaluate_test(model, test_loader, device, f'{RESULTS_DIR}/test_preds.json')

def run_xai():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if not os.path.exists('dataset/coco/test.json'):
        return
    test_loader = DataLoader(CocoDetection('dataset/coco/test', 'dataset/coco/test.json', transforms=get_transform(train=False)), batch_size=1, shuffle=True, collate_fn=collate_fn, num_workers=2)
    model = fcos_resnet50_fpn(weights_backbone="DEFAULT", num_classes=3)
    model.load_state_dict(torch.load(f'{RESULTS_DIR}/weights/best_model.pth', map_location=device, weights_only=True))
    model.to(device).eval()
    target_layer = get_target_layer(model, 'fcos')
    gradcam = GradCAM(model, target_layer)
    count = 0
    for images, targets in test_loader:
        if count >= 5:
            break
        image = images[0].unsqueeze(0).to(device)
        with torch.set_grad_enabled(True):
            heatmap = gradcam.generate(image)
        if heatmap is not None:
            overlay = overlay_heatmap(image, heatmap)
            with torch.no_grad():
                outputs = model(image)
            detections = {'boxes': outputs[0]['boxes'].cpu().numpy(), 'scores': outputs[0]['scores'].cpu().numpy(), 'labels': outputs[0]['labels'].cpu().numpy()}
            overlay = draw_detections(overlay, detections)
            overlay.save(f'{RESULTS_DIR}/explain/sample_{count}.png')
            count += 1
    gradcam.remove_hooks()

if __name__ == '__main__':
    train()
    evaluate_model()
    run_xai()

### Cell 12: Write `train_efficientdet.py`


In [ ]:
%%writefile train_efficientdet.py
import os, sys, json
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

sys.path.insert(0, os.path.dirname(__file__))
from utils.coco_dataset import CocoDetection, get_transform, collate_fn
from utils.engine import evaluate, evaluate_test, compute_confusion_matrix, save_confusion_matrix_plot
from explain.gradcam import GradCAM, get_target_layer
from explain.visualize import overlay_heatmap, draw_detections

RESULTS_DIR = 'results/efficientdet'
os.makedirs(f'{RESULTS_DIR}/explain', exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/weights', exist_ok=True)

def collate_effdet(batch):
    images, targets = zip(*batch)
    images = torch.stack(list(images))
    return images, {
        'cls': [t['labels'] for t in targets],
        'box': [t['boxes'] for t in targets],
        'img_id': [t['image_id'] for t in targets]
    }

def pad_targets(cls_list, box_list, batch_size, max_boxes=50):
    device = box_list[0].device
    cls_pad = torch.zeros(batch_size, max_boxes, dtype=torch.int64, device=device)
    box_pad = torch.zeros(batch_size, max_boxes, 4, dtype=torch.float32, device=device)
    for i in range(batch_size):
        n = min(len(cls_list[i]), max_boxes)
        cls_pad[i, :n] = cls_list[i][:n]
        box_pad[i, :n] = box_list[i][:n]
    return cls_pad, box_pad

def get_weighted_sampler(dataset, pos_weight=4.0):
    weights = []
    for i in range(len(dataset)):
        _, target = dataset[i]
        weights.append(pos_weight if len(target['boxes']) > 0 else 1.0)
    return WeightedRandomSampler(weights, len(weights), replacement=True)

def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'[EfficientDet] Device: {device}')
    from effdet import create_model
    train_dataset = CocoDetection('dataset/coco/train', 'dataset/coco/train.json', transforms=get_transform(train=True))
    val_dataset = CocoDetection('dataset/coco/val', 'dataset/coco/val.json', transforms=get_transform(train=False))
    train_loader = DataLoader(train_dataset, batch_size=8, sampler=get_weighted_sampler(train_dataset), collate_fn=collate_effdet, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)
    model = create_model('tf_efficientdet_d2', pretrained=True, num_classes=3).to(device)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
    best_map, best_epoch = 0.0, -1
    for epoch in range(1, 101):
        model.train()
        total_loss = 0
        for i, (images, batch_t) in enumerate(train_loader):
            cls_list = [c.to(device) for c in batch_t['cls']]
            box_list = [b.to(device) for b in batch_t['box']]
            images = images.to(device)
            optimizer.zero_grad()
            if scaler is not None:
                with torch.amp.autocast(device_type='cuda'):
                    cls_t, box_t = pad_targets(cls_list, box_list, images.shape[0])
                    loss_dict, _ = model(images, cls_t, box_t)
                loss = sum(loss_dict.values())
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                cls_t, box_t = pad_targets(cls_list, box_list, images.shape[0])
                loss_dict, _ = model(images, cls_t, box_t)
                loss = sum(loss_dict.values())
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item()
            if i % 50 == 0:
                print(f'  Epoch [{epoch}] Step [{i}/{len(train_loader)}] Loss: {loss.item():.4f}')
        scheduler.step()
        print(f'  Epoch [{epoch}] Avg Loss: {total_loss/len(train_loader):.4f}')
        model.eval()
        coco_eval = evaluate(model, val_loader, device)
        if coco_eval is not None and coco_eval.stats[0] > best_map:
            best_map, best_epoch = coco_eval.stats[0], epoch
            torch.save(model.state_dict(), f'{RESULTS_DIR}/weights/best_model.pth')
            print(f'  New best mAP={best_map:.4f} (epoch {epoch})')
    print(f'[EfficientDet] Best mAP: {best_map:.4f} at epoch {best_epoch}')

def evaluate_model():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    from effdet import create_model
    val_loader = DataLoader(CocoDetection('dataset/coco/val', 'dataset/coco/val.json', transforms=get_transform(train=False)), batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)
    model = create_model('tf_efficientdet_d2', pretrained=False, num_classes=3)
    model.load_state_dict(torch.load(f'{RESULTS_DIR}/weights/best_model.pth', map_location=device, weights_only=True))
    model.to(device).eval()
    coco_eval = evaluate(model, val_loader, device, output_file=f'{RESULTS_DIR}/val_preds.json')
    if coco_eval is not None:
        metrics = {
            'mAP@0.5:0.95': coco_eval.stats[0], 'mAP@0.5': coco_eval.stats[1],
            'AP_ActiveTB': coco_eval.eval['precision'][0,:,1,0,2].mean(),
            'AP_ObsoleteTB': coco_eval.eval['precision'][0,:,2,0,2].mean(),
        }
        with open(f'{RESULTS_DIR}/metrics.json', 'w') as f:
            json.dump(metrics, f, indent=2)
    confusion = compute_confusion_matrix(model, val_loader, device)
    save_confusion_matrix_plot(confusion, f'{RESULTS_DIR}/confusion_matrix.png')
    if os.path.exists('dataset/coco/test.json'):
        test_loader = DataLoader(CocoDetection('dataset/coco/test', 'dataset/coco/test.json', transforms=get_transform(train=False)), batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)
        evaluate_test(model, test_loader, device, f'{RESULTS_DIR}/test_preds.json')

def run_xai():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if not os.path.exists('dataset/coco/test.json'):
        return
    from effdet import create_model
    test_loader = DataLoader(CocoDetection('dataset/coco/test', 'dataset/coco/test.json', transforms=get_transform(train=False)), batch_size=1, shuffle=True, collate_fn=collate_fn, num_workers=2)
    model = create_model('tf_efficientdet_d2', pretrained=False, num_classes=3)
    model.load_state_dict(torch.load(f'{RESULTS_DIR}/weights/best_model.pth', map_location=device, weights_only=True))
    model.to(device).eval()
    target_layer = get_target_layer(model, 'efficientdet')
    gradcam = GradCAM(model, target_layer)
    count = 0
    for images, targets in test_loader:
        if count >= 5:
            break
        image = images[0].unsqueeze(0).to(device)
        with torch.set_grad_enabled(True):
            heatmap = gradcam.generate(image)
        if heatmap is not None:
            overlay = overlay_heatmap(image, heatmap)
            with torch.no_grad():
                outputs = model(image)
            if isinstance(outputs, list) and len(outputs) > 0:
                detections = {'boxes': outputs[0]['boxes'].cpu().numpy(), 'scores': outputs[0]['scores'].cpu().numpy(), 'labels': outputs[0]['labels'].cpu().numpy()}
                overlay = draw_detections(overlay, detections)
                overlay.save(f'{RESULTS_DIR}/explain/sample_{count}.png')
                count += 1
    gradcam.remove_hooks()

if __name__ == '__main__':
    train()
    evaluate_model()
    run_xai()

### Cell 13: Write `train_retinanet.py`


In [ ]:
%%writefile train_retinanet.py
import os, sys, json
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.models.detection import retinanet_resnet50_fpn_v2

sys.path.insert(0, os.path.dirname(__file__))
from utils.coco_dataset import CocoDetection, get_transform, collate_fn
from utils.engine import train_one_epoch, evaluate, evaluate_test, compute_confusion_matrix, save_confusion_matrix_plot
from explain.gradcam import GradCAM, get_target_layer
from explain.visualize import overlay_heatmap, draw_detections

RESULTS_DIR = 'results/retinanet'
os.makedirs(f'{RESULTS_DIR}/explain', exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/weights', exist_ok=True)

def get_weighted_sampler(dataset, pos_weight=4.0):
    weights = []
    for i in range(len(dataset)):
        _, target = dataset[i]
        weights.append(pos_weight if len(target['boxes']) > 0 else 1.0)
    return WeightedRandomSampler(weights, len(weights), replacement=True)

def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'[RetinaNet] Device: {device}')
    train_dataset = CocoDetection('dataset/coco/train', 'dataset/coco/train.json', transforms=get_transform(train=True))
    val_dataset = CocoDetection('dataset/coco/val', 'dataset/coco/val.json', transforms=get_transform(train=False))
    train_loader = DataLoader(train_dataset, batch_size=8, sampler=get_weighted_sampler(train_dataset), collate_fn=collate_fn, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)
    model = retinanet_resnet50_fpn_v2(weights_backbone="DEFAULT", num_classes=3).to(device)
    optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=0.001, momentum=0.9, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
    best_map, best_epoch = 0.0, -1
    for epoch in range(1, 101):
        train_one_epoch(model, optimizer, train_loader, device, epoch, scaler=scaler, clip_norm=1.0)
        scheduler.step()
        coco_eval = evaluate(model, val_loader, device)
        if coco_eval is not None and coco_eval.stats[0] > best_map:
            best_map, best_epoch = coco_eval.stats[0], epoch
            torch.save(model.state_dict(), f'{RESULTS_DIR}/weights/best_model.pth')
            print(f'  New best mAP={best_map:.4f} (epoch {epoch})')
    print(f'[RetinaNet] Best mAP: {best_map:.4f} at epoch {best_epoch}')

def evaluate_model():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    val_loader = DataLoader(CocoDetection('dataset/coco/val', 'dataset/coco/val.json', transforms=get_transform(train=False)), batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)
    model = retinanet_resnet50_fpn_v2(weights_backbone="DEFAULT", num_classes=3)
    model.load_state_dict(torch.load(f'{RESULTS_DIR}/weights/best_model.pth', map_location=device, weights_only=True))
    model.to(device).eval()
    coco_eval = evaluate(model, val_loader, device, output_file=f'{RESULTS_DIR}/val_preds.json')
    if coco_eval is not None:
        metrics = {
            'mAP@0.5:0.95': coco_eval.stats[0], 'mAP@0.5': coco_eval.stats[1],
            'AP_ActiveTB': coco_eval.eval['precision'][0,:,1,0,2].mean(),
            'AP_ObsoleteTB': coco_eval.eval['precision'][0,:,2,0,2].mean(),
        }
        with open(f'{RESULTS_DIR}/metrics.json', 'w') as f:
            json.dump(metrics, f, indent=2)
    confusion = compute_confusion_matrix(model, val_loader, device)
    save_confusion_matrix_plot(confusion, f'{RESULTS_DIR}/confusion_matrix.png')
    if os.path.exists('dataset/coco/test.json'):
        test_loader = DataLoader(CocoDetection('dataset/coco/test', 'dataset/coco/test.json', transforms=get_transform(train=False)), batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)
        evaluate_test(model, test_loader, device, f'{RESULTS_DIR}/test_preds.json')

def run_xai():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if not os.path.exists('dataset/coco/test.json'):
        return
    test_loader = DataLoader(CocoDetection('dataset/coco/test', 'dataset/coco/test.json', transforms=get_transform(train=False)), batch_size=1, shuffle=True, collate_fn=collate_fn, num_workers=2)
    model = retinanet_resnet50_fpn_v2(weights_backbone="DEFAULT", num_classes=3)
    model.load_state_dict(torch.load(f'{RESULTS_DIR}/weights/best_model.pth', map_location=device, weights_only=True))
    model.to(device).eval()
    target_layer = get_target_layer(model, 'retinanet')
    gradcam = GradCAM(model, target_layer)
    count = 0
    for images, targets in test_loader:
        if count >= 5:
            break
        image = images[0].unsqueeze(0).to(device)
        with torch.set_grad_enabled(True):
            heatmap = gradcam.generate(image)
        if heatmap is not None:
            overlay = overlay_heatmap(image, heatmap)
            with torch.no_grad():
                outputs = model(image)
            detections = {'boxes': outputs[0]['boxes'].cpu().numpy(), 'scores': outputs[0]['scores'].cpu().numpy(), 'labels': outputs[0]['labels'].cpu().numpy()}
            overlay = draw_detections(overlay, detections)
            overlay.save(f'{RESULTS_DIR}/explain/sample_{count}.png')
            count += 1
    gradcam.remove_hooks()

if __name__ == '__main__':
    train()
    evaluate_model()
    run_xai()

### Cell 14: Write `train_detr.py`


In [ ]:
%%writefile train_detr.py
import os, sys, json
import torch
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.models.detection import detr_resnet50
from PIL import Image, ImageDraw

sys.path.insert(0, os.path.dirname(__file__))
from utils.coco_dataset import CocoDetection, get_transform, collate_fn
from utils.engine import train_one_epoch, evaluate, evaluate_test, compute_confusion_matrix, save_confusion_matrix_plot
from explain.detr_attention import DETRAttentionExtractor

RESULTS_DIR = 'results/detr'
os.makedirs(f'{RESULTS_DIR}/explain', exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/weights', exist_ok=True)
CLASS_NAMES = {0: 'Background', 1: 'ActiveTuberculosis', 2: 'ObsoletePulmonaryTuberculosis'}
CLASS_COLORS = {1: (255, 0, 0), 2: (0, 200, 255)}

def get_weighted_sampler(dataset, pos_weight=4.0):
    weights = []
    for i in range(len(dataset)):
        _, target = dataset[i]
        weights.append(pos_weight if len(target['boxes']) > 0 else 1.0)
    return WeightedRandomSampler(weights, len(weights), replacement=True)

def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'[DETR] Device: {device}')
    train_dataset = CocoDetection('dataset/coco/train', 'dataset/coco/train.json', transforms=get_transform(train=True))
    val_dataset = CocoDetection('dataset/coco/val', 'dataset/coco/val.json', transforms=get_transform(train=False))
    train_loader = DataLoader(train_dataset, batch_size=4, sampler=get_weighted_sampler(train_dataset), collate_fn=collate_fn, num_workers=2, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)
    model = detr_resnet50(weights_backbone="DEFAULT", num_classes=3).to(device)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
    best_map, best_epoch = 0.0, -1
    for epoch in range(1, 101):
        train_one_epoch(model, optimizer, train_loader, device, epoch, scaler=scaler, clip_norm=1.0)
        scheduler.step()
        coco_eval = evaluate(model, val_loader, device)
        if coco_eval is not None and coco_eval.stats[0] > best_map:
            best_map, best_epoch = coco_eval.stats[0], epoch
            torch.save(model.state_dict(), f'{RESULTS_DIR}/weights/best_model.pth')
            print(f'  New best mAP={best_map:.4f} (epoch {epoch})')
    print(f'[DETR] Best mAP: {best_map:.4f} at epoch {best_epoch}')

def evaluate_model():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    val_loader = DataLoader(CocoDetection('dataset/coco/val', 'dataset/coco/val.json', transforms=get_transform(train=False)), batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)
    model = detr_resnet50(weights_backbone="DEFAULT", num_classes=3)
    model.load_state_dict(torch.load(f'{RESULTS_DIR}/weights/best_model.pth', map_location=device, weights_only=True))
    model.to(device).eval()
    coco_eval = evaluate(model, val_loader, device, output_file=f'{RESULTS_DIR}/val_preds.json')
    if coco_eval is not None:
        metrics = {
            'mAP@0.5:0.95': coco_eval.stats[0], 'mAP@0.5': coco_eval.stats[1],
            'AP_ActiveTB': coco_eval.eval['precision'][0,:,1,0,2].mean(),
            'AP_ObsoleteTB': coco_eval.eval['precision'][0,:,2,0,2].mean(),
        }
        with open(f'{RESULTS_DIR}/metrics.json', 'w') as f:
            json.dump(metrics, f, indent=2)
    confusion = compute_confusion_matrix(model, val_loader, device)
    save_confusion_matrix_plot(confusion, f'{RESULTS_DIR}/confusion_matrix.png')
    if os.path.exists('dataset/coco/test.json'):
        test_loader = DataLoader(CocoDetection('dataset/coco/test', 'dataset/coco/test.json', transforms=get_transform(train=False)), batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)
        evaluate_test(model, test_loader, device, f'{RESULTS_DIR}/test_preds.json')

def run_xai():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if not os.path.exists('dataset/coco/test.json'):
        return
    test_loader = DataLoader(CocoDetection('dataset/coco/test', 'dataset/coco/test.json', transforms=get_transform(train=False)), batch_size=1, shuffle=True, collate_fn=collate_fn, num_workers=2)
    model = detr_resnet50(weights_backbone="DEFAULT", num_classes=3)
    model.load_state_dict(torch.load(f'{RESULTS_DIR}/weights/best_model.pth', map_location=device, weights_only=True))
    model.to(device).eval()
    count = 0
    for images, targets in test_loader:
        if count >= 5:
            break
        image = images[0].unsqueeze(0).to(device)
        extractor = DETRAttentionExtractor(model)
        heatmaps, meta = extractor.extract(image)
        extractor.remove_hook()
        if heatmaps and len(heatmaps) > 0:
            best_idx = max(range(len(meta)), key=lambda i: meta[i]['score'])
            if best_idx < len(heatmaps):
                img_np = image.squeeze(0).permute(1,2,0).cpu().numpy()
                img_np = (img_np * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406]))
                img_np = np.clip(img_np*255, 0, 255).astype(np.uint8)
                import matplotlib
                matplotlib.use('Agg')
                import matplotlib.pyplot as plt
                fig, axes = plt.subplots(1, 2, figsize=(12, 6))
                draw_img = Image.fromarray(img_np.copy())
                draw = ImageDraw.Draw(draw_img)
                with torch.no_grad():
                    outputs = model(image)
                for box, sc, lb in zip(outputs[0]['boxes'].cpu().numpy(), outputs[0]['scores'].cpu().numpy(), outputs[0]['labels'].cpu().numpy()):
                    if sc >= 0.3:
                        x1, y1, x2, y2 = box[:4]
                        draw.rectangle([x1, y1, x2, y2], outline=CLASS_COLORS.get(int(lb), (255,255,0)), width=3)
                        draw.text((x1+2, y1-16), f'{CLASS_NAMES.get(int(lb), str(lb))}: {sc:.2f}', fill=CLASS_COLORS.get(int(lb), (255,255,0)))
                axes[0].imshow(np.array(draw_img))
                axes[0].set_title('DETR - Detections')
                attn = heatmaps[best_idx]
                attn_resized = np.array(Image.fromarray((attn*255).astype(np.uint8)).resize((img_np.shape[1], img_np.shape[0]), Image.BILINEAR)) / 255.0
                cmap = plt.get_cmap('jet')
                combined = (1-0.5) * img_np/255 + 0.5 * cmap(attn_resized)[:,:,:3]
                axes[1].imshow(np.clip(combined, 0, 1))
                axes[1].set_title(f'Cross-Attention (score={meta[best_idx]["score"]:.2f})')
                for ax in axes:
                    ax.axis('off')
                plt.tight_layout()
                plt.savefig(f'{RESULTS_DIR}/explain/sample_{count}.png', dpi=150, bbox_inches='tight')
                plt.close()
                count += 1
    print('[DETR] XAI complete.')

if __name__ == '__main__':
    train()
    evaluate_model()
    run_xai()

In [ ]:
# Cell 15: Convert Annotations + Run EDA
%cd /content/OBJECT_DETECTION
!python convert.py --data-root Images --output-dir dataset
!python eda.py

from IPython.display import Image as DImg, display
for fname in ['tag_distribution.png', 'bbox_class_distribution.png', 'bbox_spatial_heatmap.png',
              'bbox_size_distribution.png', 'bbox_aspect_ratio.png', 'boxes_per_image.png', 'sample_grid.png']:
    path = f'results/eda/{fname}'
    if os.path.exists(path):
        display(DImg(path))

In [ ]:
# Cell 16: Train FCOS
%cd /content/OBJECT_DETECTION
!python train_fcos.py 2>&1 | tee results/fcos/training_log.txt
print('FCOS complete.')

In [ ]:
# Cell 17: Train EfficientDet-D2
%cd /content/OBJECT_DETECTION
!python train_efficientdet.py 2>&1 | tee results/efficientdet/training_log.txt
print('EfficientDet complete.')

In [ ]:
# Cell 18: Train RetinaNet
%cd /content/OBJECT_DETECTION
!python train_retinanet.py 2>&1 | tee results/retinanet/training_log.txt
print('RetinaNet complete.')

In [ ]:
# Cell 19: Train DETR
%cd /content/OBJECT_DETECTION
!python train_detr.py 2>&1 | tee results/detr/training_log.txt
print('DETR complete.')

In [ ]:
# Cell 20: Model Comparison
%cd /content/OBJECT_DETECTION
import os, json, pandas as pd, matplotlib.pyplot as plt

models = ['fcos', 'efficientdet', 'retinanet', 'detr']
labels = ['FCOS', 'EfficientDet-D2', 'RetinaNet', 'DETR']

all_metrics = {}
for m in models:
    p = f'results/{m}/metrics.json'
    if os.path.exists(p):
        with open(p) as f:
            all_metrics[m] = json.load(f)

df = pd.DataFrame(all_metrics).T
df.index = labels
print('\n=== MODEL COMPARISON ===')
print(df.round(4).to_string())
df.round(4).to_csv('results/comparison.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if 'mAP@0.5:0.95' in df.columns:
    df['mAP@0.5:0.95'].plot(kind='bar', ax=axes[0], color=['#4C72B0','#DD8452','#55A868','#C44E52'])
    axes[0].set_ylabel('mAP@0.5:0.95')
    axes[0].set_title('Detection Performance')
if 'AP_ActiveTB' in df.columns and 'AP_ObsoleteTB' in df.columns:
    df[['AP_ActiveTB', 'AP_ObsoleteTB']].plot(kind='bar', ax=axes[1])
    axes[1].set_ylabel('AP')
    axes[1].set_title('Per-Class AP')
plt.tight_layout()
plt.savefig('results/comparison_barplot.png', dpi=150)
plt.show()

In [ ]:
# Cell 21: Confusion Matrices + XAI
%cd /content/OBJECT_DETECTION
from IPython.display import Image as DImg, display

for name, label in zip(['fcos','efficientdet','retinanet','detr'],
                       ['FCOS','EfficientDet-D2','RetinaNet','DETR']):
    cm = f'results/{name}/confusion_matrix.png'
    if os.path.exists(cm):
        print(f'\n{label}:')
        display(DImg(cm))
    xdir = f'results/{name}/explain'
    if os.path.isdir(xdir):
        for s in sorted(os.listdir(xdir))[:2]:
            display(DImg(os.path.join(xdir, s)))